# 02 · Chunking

Compare the **proposed** strategy (B: section-aware, ~400 tokens, 60 overlap, context header) with the **baseline** (A: recursive split over the whole text, 800/120). The evaluation in Phase 7 decides which wins; here we check that both behave as designed.

In [1]:
import sys, uuid, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))
from tests.fixtures.samples import build_samples
SAMPLES = build_samples(ROOT / "data" / "samples")
for name, path in SAMPLES.items():
    print(f"{name:14s} {path.name:22s} {path.stat().st_size:>7} bytes")

policy_pdf     leave_policy.pdf          6780 bytes
scanned_pdf    scanned.pdf            1506536 bytes
handbook_docx  it_security.docx         36767 bytes
slides_pptx    onboarding.pptx          29385 bytes
notes_md       faq.md                     128 bytes
urdu_txt       urdu_notice.txt            139 bytes
empty_pdf      empty.pdf                    0 bytes
fake_pdf       invoice.pdf                204 bytes


## 1. Real sample: the leave-policy PDF

In [2]:
from app.ingestion.pipeline import process_file
from app.ingestion.chunking import PROPOSED, BASELINE
data = SAMPLES["policy_pdf"].read_bytes()
for cfg in (PROPOSED, BASELINE):
    res = process_file("leave_policy.pdf", data, document_id=str(uuid.uuid4()), config=cfg)
    print(f"\n=== config {cfg.name}: {cfg.strategy}, size {cfg.size}, overlap {cfg.overlap} -> {len(res.chunks)} chunks")
    for c in res.chunks:
        print(f"[{c.position}] pages {c.page_start}-{c.page_end} {c.kind:5s} {c.token_count:>3} tok | {c.section}")


=== config B: section, size 400, overlap 60 -> 4 chunks
[0] pages 1-1 text   46 tok | 1. Annual Leave
[1] pages 2-2 text   33 tok | 2. Sick Leave
[2] pages 2-2 table  46 tok | 2. Sick Leave
[3] pages 3-3 text   27 tok | 3. Remote Work



=== config A: fixed, size 800, overlap 120 -> 1 chunks
[0] pages 1-3 text  152 tok | 1. Annual Leave


## 2. Metadata stored with every chunk (requirement: document id, file name, page/section, position)

In [3]:
from dataclasses import asdict
res = process_file("leave_policy.pdf", data, document_id=str(uuid.uuid4()), config=PROPOSED)
print(json.dumps(asdict(res.chunks[1]), indent=2, ensure_ascii=False))

{
  "id": "98369bcf-7728-5310-b7f3-2fcd44b80c16",
  "document_id": "137a91c3-a467-4264-a37e-ad917416a56e",
  "filename": "leave_policy.pdf",
  "position": 1,
  "text": "Employees are entitled to 10 paid sick days per year. A medical certificate is required for any absence longer than three consecutive days. Sick leave cannot be carried over.",
  "embed_text": "leave_policy.pdf > 2. Sick Leave\n\nEmployees are entitled to 10 paid sick days per year. A medical certificate is required for any absence longer than three consecutive days. Sick leave cannot be carried over.",
  "page_start": 2,
  "page_end": 2,
  "section": "2. Sick Leave",
  "token_count": 33,
  "kind": "text",
  "chunking_config": "B"
}


## 3. A longer document: sizes, overlap and section boundaries
Synthetic text of ~1,500 tokens over two pages and two sections.

In [4]:
from app.ingestion.extractors.base import TextUnit
from app.ingestion.chunking import chunk_document
sentences = [f"Rule {i} says employees must follow procedure number {i} carefully." for i in range(120)]
units = [TextUnit(" ".join(sentences[:60]), page=1, section="Rules > Part A"),
         TextUnit(" ".join(sentences[60:]), page=2, section="Rules > Part B")]
for cfg in (PROPOSED, BASELINE):
    chunks = chunk_document(units, document_id=str(uuid.uuid4()), filename="rules.pdf", config=cfg)
    print(f"config {cfg.name}: {len(chunks)} chunks, tokens = {[c.token_count for c in chunks]}, "
          f"pages = {[(c.page_start, c.page_end) for c in chunks]}")

config B: 6 chunks, tokens = [390, 390, 104, 390, 390, 104], pages = [(1, 1), (1, 1), (1, 1), (2, 2), (2, 2), (2, 2)]
config A: 2 chunks, tokens = [780, 780], pages = [(1, 1), (2, 2)]


In [5]:
b = chunk_document(units, document_id=str(uuid.uuid4()), filename="rules.pdf", config=PROPOSED)
print("end of chunk 0  :", b[0].text[-120:])
print("start of chunk 1:", b[1].text[:120])
print("\nB never mixes sections:", all(not ("Rule 59 " in c.text and "Rule 60 " in c.text) for c in b))
print("\nembedded text of chunk 0 starts with the header:\n", b[0].embed_text[:80])

end of chunk 0  : s employees must follow procedure number 28 carefully. Rule 29 says employees must follow procedure number 29 carefully.
start of chunk 1: Rule 26 says employees must follow procedure number 26 carefully. Rule 27 says employees must follow procedure number 27

B never mixes sections: True

embedded text of chunk 0 starts with the header:
 rules.pdf > Rules > Part A

Rule 0 says employees must follow procedure number 0
